In [ ]:
print('-----THANK YOU FOR REVIEWING MY CODE---------------------')

# **DDOS evaluation dataset using ML and DL with EDA**

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import seaborn as sn
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.metrics import confusion_matrix
import numpy as np
from keras.models import  Sequential
from keras.layers import Dense
import keras.activations,keras.metrics,keras.losses


data=pd.read_csv('/kaggle/input/ddos-evaluation-dataset-cic-ddos2019/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
print(data.columns)
print(data.info())
print(data.isna().sum())

# **OUTLIER Information of number column**

In [ ]:
for i in data.select_dtypes(include='number').columns.values:
    sn.boxplot(data[i])
    plt.show()

# **LABEL Encoding of the columns**

In [ ]:
lab=LabelEncoder()
for i in data.select_dtypes(include='object').columns.values:
    data[i]=lab.fit_transform(data[i])

print(data.info())
print(data.isna().sum())

# **outlier detection of each column**

In [ ]:
outlier={}
final_col=[]
for i in data.columns.values:
    data['z-scores']=(data[i]-data[i].mean())/data[i].std()
    outliers=np.abs(data['z-scores']) >3
    print(f'The number of outliers in column for  {i}  ',outliers.sum())
    outlier[i]=outliers.sum()

for col,val in outlier.items():
    if val >0:
        final_col.append(col)

# **Outlier deduction of each column**

In [ ]:
print(len(final_col))
print(len(data))
for i in final_col[:37]:
    thres=3
    upper=data[i].mean()+thres*data[i].std()
    lower=data[i].mean()-thres*data[i].std()
    data=data[(data[i]>lower)&(data[i]<upper)]
print(len(data))


# **Pie chart representation of each column**

In [ ]:
for i in data.columns.values:
    if len(data[i].value_counts().values) <=5:
        values=data[i].value_counts().values
        index=data[i].value_counts().index
        plt.pie(values, labels=index, autopct='%1.1f%%')
        plt.title(f'{i} column values')
        plt.legend()
        plt.show()

# **The correlation representation of columns using heatmap**

In [ ]:
correlation_matrix = data.corr()
sn.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.show()


plt.figure(figsize=(17, 6))
corr = data.corr(method='spearman')
my_m = np.triu(corr)
sn.heatmap(corr, mask=my_m, annot=True, cmap="Set2")
plt.show()

# **Different EDa techniques to unerstand the data**

In [ ]:
correlation_matrix = data.corr()
sn.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.show()

# **MODEL BUILDING USING DIFFERENT ML ALGORITHMS**

In [ ]:
for i in final_col[:37]:
    print(i)

x=data[[' Timestamp', ' Flow Duration',
       ' Total Fwd Packets', ' Total Backward Packets',
       'Total Length of Fwd Packets', ' Total Length of Bwd Packets',
       ' Fwd Packet Length Max', ' Fwd Packet Length Min',
       ' Fwd Packet Length Mean', ' Fwd Packet Length Std',
       'Bwd Packet Length Max', ' Bwd Packet Length Min',
       ' Bwd Packet Length Mean', ' Bwd Packet Length Std',
       'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count',
        'Idle Mean', ' Idle Std',
       ' Idle Max', ' Idle Min']]
y=data[' Label']

smote=SMOTE(sampling_strategy='auto')
X,Y=smote.fit_resample(x,y)
print(Y.value_counts())


x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.5)
lr = LogisticRegression(max_iter=200)
lr.fit(x_train, y_train)
pred=lr.predict(x_test)
print('The confusion matrix ',confusion_matrix(y_test,pred))
print('The logistic regression: ', lr.score(x_test, y_test))

xgb = XGBClassifier()
xgb.fit(x_train, y_train)
pred=xgb.predict(x_test)
print('The confusion matrix ',confusion_matrix(y_test,pred))
print("the Xgb : ", xgb.score(x_test, y_test))

lgb = LGBMClassifier()
lgb.fit(x_train, y_train)
pred=lgb.predict(x_test)
print('The confusion matrix ',confusion_matrix(y_test,pred))
print('The LGB', lgb.score(x_test, y_test))

tree = DecisionTreeClassifier(criterion='gini', max_depth=1)
tree.fit(x_train, y_train)
pred=tree.predict(x_test)
print('The confusion matrix ',confusion_matrix(y_test,pred))
print('Dtree ', tree.score(x_test,y_test))

rforest = RandomForestClassifier(criterion='gini')
rforest.fit(x_train, y_train)
pred=rforest.predict(x_test)
print('The confusion matrix ',confusion_matrix(y_test,pred))
print('The random forest: ', rforest.score(x_test, y_test))

adb = AdaBoostClassifier()
adb.fit(x_train, y_train)
pred=adb.predict(x_test)
print('The confusion matrix ',confusion_matrix(y_test,pred))
print('the adb ', adb.score(x_test, y_test))

grb = GradientBoostingClassifier()
grb.fit(x_train, y_train)
pred=grb.predict(x_test)
print('The confusion matrix ',confusion_matrix(y_test,pred))
print('Gradient boosting ', grb.score(x_test, y_test))

bag = BaggingClassifier()
bag.fit(x_train, y_train)
pred=bag.predict(x_test)
print('The confusion matrix ',confusion_matrix(y_test,pred))
print('Bagging', bag.score(x_test, y_test))

# **Classification using DEEP learning**

In [ ]:
y_val=pd.get_dummies(Y)
x_tr,x_te,y_tr,y_te=train_test_split(X,y_val)

model=Sequential()
model.add(Dense(units=X.shape[1],input_dim=X.shape[1],activation=keras.activations.relu))
model.add(Dense(units=X.shape[1],activation=keras.activations.relu))
model.add(Dense(units=X.shape[1],activation=keras.activations.sigmoid))
model.add(Dense(units=X.shape[1],activation=keras.activations.sigmoid))
model.add(Dense(units=y_val.shape[1],activation=keras.activations.sigmoid))
model.compile(optimizer='rmsprop',loss=keras.losses.binary_crossentropy,metrics='accuracy')
histo=model.fit(x_tr,y_tr,batch_size=50,epochs=15,verbose=True)
plt.plot(histo.history['accuracy'], label='training accuracy', marker='o', color='red')
plt.plot(histo.history['loss'], label='loss', marker='o', color='darkblue')
plt.title('Training Vs  Validation accuracy with adam rmsprop')
plt.legend()
plt.show()


models1=Sequential()
models1.add(Dense(units=X.shape[1],input_dim=X.shape[1],activation=keras.activations.relu))
models1.add(Dense(units=X.shape[1],activation=keras.activations.relu))
models1.add(Dense(units=X.shape[1],activation=keras.activations.relu))
models1.add(Dense(units=X.shape[1],activation=keras.activations.sigmoid))
models1.add(Dense(units=X.shape[1],activation=keras.activations.sigmoid))
models1.add(Dense(units=y_val.shape[1],activation=keras.activations.sigmoid))
models1.compile(optimizer='adam',loss=keras.losses.binary_crossentropy,metrics='accuracy')
hist=models1.fit(x_tr,y_tr,batch_size=20,epochs=15)

plt.plot(hist.history['accuracy'], label='training accuracy', marker='o', color='red')
plt.plot(hist.history['loss'], label='loss', marker='o', color='darkblue')
plt.title('Training Vs  Validation accuracy with adam optimizer')
plt.legend()
plt.show()

**